# Migration Plan Generator

Pure **renderer** — reads a migration plan JSON file and generates a branded HTML report.

**Usage:** Set `PLAN_JSON_PATH` in the next cell to the JSON file path, then **Run All**.

The JSON is produced by the **migration-expert** skill (CoCoDuo). This notebook contains **zero business logic** — it only renders.

In [ ]:
PLAN_JSON_FILE = "migration_plan_assess-006-udr.json"

In [ ]:
import json, os, re, glob

ws_matches = glob.glob('/workspace/*/Migration_Plan_Generator.ipynb')
ws_root = os.path.dirname(ws_matches[0]) if ws_matches else '/workspace'
json_path = os.path.join(ws_root, 'tmp', PLAN_JSON_FILE)

if not os.path.exists(json_path):
    raise FileNotFoundError(f'Plan JSON not found at {json_path}. Run the migration-expert skill first to generate it.')

with open(json_path, 'r') as f:
    plan = json.load(f)

meta = plan['metadata']
print(f"Loaded plan for: {meta['customer_name']} ({meta['assessment_id']})")
print(f"Generated: {meta['generated_date']} by {meta['generated_by']}")

In [ ]:
TOOL_DB = {
    'Airbyte': {'sf_fit':8,'ease':7,'enterprise':5,'cost':9,'tag':'Cost-sensitive, open-source',
        'strengths':['Open-source, no per-row pricing','350+ connectors, growing fast','Self-hosted or cloud options','Extensible SDK for custom connectors'],
        'watchouts':['Self-hosted requires DevOps','Some connectors less mature','Limited built-in monitoring'],
        'capabilities':['ELT','Open Source','CDC','Cloud + Self-hosted'],
        'key_facts':[('350+','Native connectors'),('Hours','Time to value'),('50+','Dest. supported')],
        'deploy_model':[('Self-hosted','Open source'),('Airbyte Cloud','Managed SaaS')],
        'source_compat':['Cloud DBs (RDS, Aurora)','SaaS (Salesforce, HubSpot)','APIs / Webhooks','Files (S3, GCS, SFTP)'],
        'pricing':[('Open Source (self-hosted)','$0'),('Cloud','$1/credit (~$300+/mo')],
    },
    'Fivetran': {'sf_fit':9,'ease':9,'enterprise':8,'cost':4,'tag':'Small teams, zero-ops SaaS',
        'strengths':['500+ connectors, gold-standard reliability','Zero-maintenance, fully managed','Auto schema detection','Excellent Snowflake integration'],
        'watchouts':['Per-MAR pricing punishes volume','Very expensive at scale ($100K-$500K+)','Limited transformation'],
        'capabilities':['ELT','Managed SaaS','CDC','Auto-schema'],
        'key_facts':[('500+','Native connectors'),('Minutes','Time to value'),('99.9%','Uptime SLA')],
        'deploy_model':[('Fully managed','Zero-ops SaaS'),('Hybrid','Fivetran HVR')],
        'source_compat':['Cloud DBs (RDS, Aurora)','SaaS (Salesforce, HubSpot)','Oracle / SQL Server','SAP / Teradata','Files / APIs'],
        'pricing':[('Free','$0 (limited)'),('Starter','~$2,000/mo'),('Enterprise','Custom ($5K-$50K+/mo)')],
    },
    'Matillion': {'sf_fit':9,'ease':7,'enterprise':7,'cost':6,'tag':'ELT + transform in one tool',
        'strengths':['Snowflake-native pushdown SQL','Visual builder + code editor','Loading + transformation combined','Good CI/CD and Git integration'],
        'watchouts':['Credit pricing unpredictable','Steeper learning curve','Fewer SaaS connectors'],
        'capabilities':['ELT push-down','Cloud SaaS','Git-native'],
        'key_facts':[('100+','Native connectors'),('Days','Time to value')],
        'deploy_model':[('Push-down','Executes in-warehouse'),('AWS/Azure/GCP','Cloud deployment')],
        'source_compat':['Cloud DBs (RDS, Aurora)','SaaS (Salesforce, HubSpot)','Oracle / SQL Server','SAP / Teradata','Mainframe / COBOL'],
        'pricing':[('Starter','~$2,000/mo'),('Growth','~$6,000/mo'),('Enterprise','Custom')],
    },
    'Informatica Cloud (IDMC)': {'sf_fit':7,'ease':5,'enterprise':9,'cost':3,'tag':'Enterprise legacy, SAP, mainframe',
        'strengths':['Broadest connector library (SAP, mainframe)','Enterprise governance, MDM, DQ','Legacy migration heritage','CLAIRE AI for mapping'],
        'watchouts':['Very expensive ($300K-$1M+)','Complex UI, steep learning curve','Vendor lock-in, slow innovation'],
        'capabilities':['ETL + ELT hybrid','Secure Agent','MDM / DQ'],
        'key_facts':[('200+','Native connectors'),('Months','Time to value')],
        'deploy_model':[('IPU-based','Pricing model'),('Hybrid','Cloud + on-prem Agent')],
        'source_compat':['Cloud DBs (RDS, Aurora)','SaaS (Salesforce, HubSpot)','Oracle / SQL Server','SAP / Teradata','Mainframe / COBOL'],
        'pricing':[('Base IDMC','~$2,000/mo'),('With DQ + MDM','~$8,000-$50,000/mo'),('Enterprise','Custom (IPU)')],
    },
    'Hevo Data': {'sf_fit':7,'ease':8,'enterprise':5,'cost':7,'tag':'Mid-market, simple managed',
        'strengths':['Simple UI, fast setup','Transparent per-event pricing','Good CDC support','Responsive support'],
        'watchouts':['Fewer enterprise connectors','Less mature ecosystem','Limited transformation'],
        'capabilities':['ELT','Managed SaaS','CDC'],
        'key_facts':[('150+','Connectors'),('Minutes','Time to value')],
        'deploy_model':[('Fully managed','Cloud SaaS')],
        'source_compat':['Cloud DBs','SaaS (Salesforce, HubSpot)','APIs / Webhooks'],
        'pricing':[('Free','$0 (limited)'),('Starter','~$239/mo'),('Business','Custom')],
    },
    'Stitch Data': {'sf_fit':5,'ease':7,'enterprise':3,'cost':6,'tag':'Simple SaaS extraction',
        'strengths':['Simple setup','Low volume pricing'],
        'watchouts':['Frequent sync failures','Poor error handling','Limited connectors'],
        'capabilities':['ELT','Managed SaaS'],
        'key_facts':[('130+','Connectors'),('Minutes','Time to value')],
        'deploy_model':[('Fully managed','Cloud SaaS')],
        'source_compat':['SaaS apps','Cloud DBs','APIs'],
        'pricing':[('Standard','~$100/mo'),('Advanced','~$1,250/mo')],
    },
    'AWS Glue': {'sf_fit':5,'ease':4,'enterprise':6,'cost':5,'tag':'AWS-native PySpark ETL',
        'strengths':['Serverless','AWS integration','No infrastructure to manage'],
        'watchouts':['Slow startup, painful debugging','Unpredictable costs','PySpark overhead for simple SQL'],
        'capabilities':['ETL','Serverless','PySpark'],
        'key_facts':[('N/A','Native connectors'),('Weeks','Time to value')],
        'deploy_model':[('Serverless','AWS-managed'),('AWS-only','Single cloud')],
        'source_compat':['S3 / Data Lake','RDS / Redshift','JDBC sources'],
        'pricing':[('DPU-hour','~$0.44/DPU-hr')],
    },
    'Azure Data Factory (ADF)': {'sf_fit':6,'ease':7,'enterprise':7,'cost':7,'tag':'Azure-native orchestration',
        'strengths':['Azure integration','Visual pipelines','Managed IR for hybrid'],
        'watchouts':['Snowflake connector not native','Limited transformation','Activity-based pricing'],
        'capabilities':['ETL','Azure-native','Hybrid IR'],
        'key_facts':[('90+','Connectors'),('Weeks','Time to value')],
        'deploy_model':[('Pay-per-run','Activity pricing'),('Azure-only','Single cloud')],
        'source_compat':['Azure SQL / Synapse','SaaS via connectors','Oracle / SQL Server','SAP / Teradata','On-prem via IR'],
        'pricing':[('Orchestration','~$0.001/run'),('Data movement','~$0.25/DIU-hr'),('Typical mid-project','~$200-800/mo')],
    },
    'Custom Scripts (Python/Spark)': {'sf_fit':5,'ease':4,'enterprise':2,'cost':9,'tag':'Custom code, no licensing',
        'strengths':['Full flexibility','No licensing cost','Any source/target'],
        'watchouts':['No monitoring or alerting','Fragile, undocumented','Hard to maintain and scale'],
        'capabilities':['Custom code','Any language','Self-managed'],
        'key_facts':[('Unlimited','Flexibility'),('Weeks+','Time to value')],
        'deploy_model':[('Self-managed','Any infra')],
        'source_compat':['Anything with an API or driver'],
        'pricing':[('Licensing','$0'),('Infrastructure + labor','Variable')],
    },
    'SSIS (SQL Server Integration Services)': {'sf_fit':3,'ease':5,'enterprise':6,'cost':5,'tag':'SQL Server native ETL',
        'strengths':['Native SQL Server integration','Mature error handling'],
        'watchouts':['Windows-only','No Snowflake-native support','Legacy, declining'],
        'capabilities':['ETL','On-prem','SQL Server native'],
        'key_facts':[('SQL Server','Primary target'),('Years','Ecosystem maturity')],
        'deploy_model':[('On-prem','Windows Server'),('SQL Agent','Job scheduling')],
        'source_compat':['SQL Server','Oracle','Flat files','ODBC / OLE DB'],
        'pricing':[('Included','With SQL Server license')],
    },
    'Informatica PowerCenter': {'sf_fit':4,'ease':4,'enterprise':8,'cost':2,'tag':'Legacy enterprise ETL',
        'strengths':['Mature','Feature-rich for on-prem'],
        'watchouts':['End-of-life trajectory','Very expensive','Hard to hire talent'],
        'capabilities':['ETL','On-prem','Enterprise'],
        'key_facts':[('200+','Connectors'),('Months','Time to value')],
        'deploy_model':[('On-prem','Server-based'),('Legacy','End-of-life track')],
        'source_compat':['Oracle','SQL Server','SAP','Mainframe','Teradata'],
        'pricing':[('Enterprise license','$300K-$1M+/yr')],
    },
    'Ab Initio': {'sf_fit':3,'ease':3,'enterprise':7,'cost':1,'tag':'Legacy parallel ETL',
        'strengths':['High performance parallel processing'],
        'watchouts':['Extremely expensive ($500K-$1M+)','Impossible to hire talent','Total vendor lock-in'],
        'capabilities':['ETL','Parallel processing','On-prem'],
        'key_facts':[('Massive','Parallel scale'),('Months','Time to value')],
        'deploy_model':[('On-prem','Server clusters')],
        'source_compat':['Mainframe','Oracle','Files'],
        'pricing':[('Enterprise','$500K-$1M+/yr')],
    },
    'dbt-core': {'sf_fit':9,'ease':7,'enterprise':5,'cost':10,'tag':'Free, SQL-first transformation',
        'strengths':['Free and open-source','SQL-first for analysts','Excellent testing & docs','Massive community'],
        'watchouts':['No built-in scheduler','CLI only, no visual UI','Enterprise features need dbt Cloud'],
        'capabilities':['SQL-first','Open Source','Jinja templating','Git-native'],
        'key_facts':[('50K+','GitHub stars'),('Hours','Time to value'),('1000+','Community packages')],
        'deploy_model':[('CLI','Local / CI/CD'),('Open Source','Free forever')],
        'source_compat':['Snowflake','BigQuery','Redshift','Databricks','Postgres'],
        'pricing':[('dbt-core','$0 (open source)'),('dbt Cloud Dev','$100/seat/mo'),('dbt Cloud Enterprise','$500/seat/mo')],
    },
    'dbt Cloud': {'sf_fit':9,'ease':9,'enterprise':8,'cost':6,'tag':'Managed CI/CD for dbt',
        'strengths':['Built-in scheduler, CI/CD, IDE','Snowflake-optimized','Managed docs & freshness','Slim CI for changed models'],
        'watchouts':['$100-$500/seat/month','Overkill for solo devs','Some features locked to Enterprise'],
        'capabilities':['SQL-first','Managed SaaS','CI/CD','IDE'],
        'key_facts':[('Built-in','Scheduler + IDE'),('Minutes','Time to deploy')],
        'deploy_model':[('Managed SaaS','Fully hosted'),('Multi-tenant','Shared infra')],
        'source_compat':['Snowflake','BigQuery','Redshift','Databricks'],
        'pricing':[('Developer','$100/seat/mo'),('Team','$100/seat/mo'),('Enterprise','Custom ($500+/seat/mo)')],
    },
    'dbt (data build tool)': {'sf_fit':9,'ease':7,'enterprise':5,'cost':10,'tag':'Free, SQL-first transformation',
        'strengths':['Free and open-source','SQL-first for analysts','Excellent testing & docs','Massive community'],
        'watchouts':['No built-in scheduler','CLI only','Enterprise features need dbt Cloud'],
        'capabilities':['SQL-first','Open Source','Jinja templating'],
        'key_facts':[('50K+','GitHub stars'),('Hours','Time to value')],
        'deploy_model':[('CLI','Local / CI/CD'),('Open Source','Free')],
        'source_compat':['Snowflake','BigQuery','Redshift','Databricks','Postgres'],
        'pricing':[('dbt-core','$0 (open source)')],
    },
    'Coalesce': {'sf_fit':9,'ease':9,'enterprise':6,'cost':6,'tag':'Visual transformation builder',
        'strengths':['Drag-and-drop visual builder','Snowflake-native SQL generation','Column-level lineage','Good for mixed-skill teams'],
        'watchouts':['Smaller community than dbt','Snowflake-only','Less Python flexibility'],
        'capabilities':['Visual builder','Snowflake-native','Column lineage','Git-native'],
        'key_facts':[('Visual','No-code / low-code'),('Hours','Time to value'),('Full','Column lineage')],
        'deploy_model':[('Managed SaaS','Fully hosted'),('Snowflake-only','Single target')],
        'source_compat':['Snowflake (exclusive)'],
        'pricing':[('Starter','~$500/mo'),('Professional','~$1,500/mo'),('Enterprise','Custom')],
    },
    'SQLMesh': {'sf_fit':8,'ease':7,'enterprise':5,'cost':10,'tag':'dbt alternative with environments',
        'strengths':['Built-in virtual environments','Change categorization','dbt compatibility mode','Free and open-source'],
        'watchouts':['Much smaller community','Less ecosystem support','Newer, less battle-tested'],
        'capabilities':['SQL-first','Open Source','Virtual envs'],
        'key_facts':[('Built-in','Virtual environments'),('Hours','Time to value')],
        'deploy_model':[('CLI','Local / CI/CD'),('Open Source','Free')],
        'source_compat':['Snowflake','BigQuery','Postgres','DuckDB'],
        'pricing':[('Open Source','$0'),('Tobiko Cloud','Custom')],
    },
    'Snowpark (Python / Java / Scala)': {'sf_fit':10,'ease':6,'enterprise':7,'cost':10,'tag':'Native Snowflake code execution',
        'strengths':['Runs in Snowflake, no data movement','Python/Java/Scala support','ML feature engineering','Integrates with Snowpark ML'],
        'watchouts':['Steeper learning curve','Debugging challenges','Overkill for simple SQL'],
        'capabilities':['Native Snowflake','Python/Java/Scala','ML-ready'],
        'key_facts':[('Zero','Data movement'),('Native','Runs in Snowflake')],
        'deploy_model':[('Native','Snowflake compute'),('Warehouse','Credit-based')],
        'source_compat':['Snowflake (exclusive, runs inside)'],
        'pricing':[('Included','Warehouse credits only')],
    },
    'Stored Procedures (in-database)': {'sf_fit':7,'ease':6,'enterprise':5,'cost':10,'tag':'Native database logic',
        'strengths':['No external tool needed','Runs in Snowflake','Good for simple procedural logic'],
        'watchouts':['Less testable than dbt','No lineage or documentation','Hard to version control'],
        'capabilities':['In-database','SQL / JS','Zero cost'],
        'key_facts':[('Native','No external tool'),('N/A','No licensing')],
        'deploy_model':[('In-database','Snowflake native')],
        'source_compat':['Snowflake (exclusive)'],
        'pricing':[('Included','Warehouse credits only')],
    },
    'Tableau': {'sf_fit':8,'ease':7,'enterprise':9,'cost':5,'tag':'Enterprise visualization leader',
        'strengths':['Industry-leading visualizations','Large talent pool','Snowflake Live Query','Enterprise governance'],
        'watchouts':['Expensive ($70+/user/mo)','Complex for self-service','Extracts create data sprawl'],
        'capabilities':['Enterprise BI','Visualizations','Live Query','Governed'],
        'key_facts':[('1M+','Global users'),('Days','Time to value'),('#1','BI market share')],
        'deploy_model':[('Tableau Cloud','Managed SaaS'),('Tableau Server','Self-hosted')],
        'source_compat':['Snowflake (Live Query)','All major databases','Files','APIs'],
        'pricing':[('Creator','$75/user/mo'),('Explorer','$42/user/mo'),('Viewer','$15/user/mo')],
    },
    'Sigma Computing': {'sf_fit':10,'ease':9,'enterprise':7,'cost':7,'tag':'Snowflake-native, self-service',
        'strengths':['Built natively for Snowflake','Spreadsheet UX for business users','No extracts, always live','Excellent self-service'],
        'watchouts':['Smaller ecosystem','Less pixel-perfect reporting','Growing enterprise track record'],
        'capabilities':['Snowflake-native','Spreadsheet UX','Always live','Self-service'],
        'key_facts':[('Zero','Extracts needed'),('Hours','Time to value'),('Live','Always real-time')],
        'deploy_model':[('Managed SaaS','Fully hosted'),('Snowflake-native','Direct query')],
        'source_compat':['Snowflake (native)','BigQuery','Redshift','Databricks'],
        'pricing':[('Essential','Contact sales'),('Professional','Contact sales'),('Enterprise','Custom')],
    },
    'Power BI': {'sf_fit':6,'ease':8,'enterprise':8,'cost':8,'tag':'Affordable, Microsoft ecosystem',
        'strengths':['$10/user/mo, very affordable','Strong Microsoft integration','Large community','AI Copilot'],
        'watchouts':['DirectQuery perf limitations','Import copies data','Less Snowflake-native'],
        'capabilities':['Microsoft native','AI Copilot','DirectQuery','Import mode'],
        'key_facts':[('$10','Per user/mo'),('Hours','Time to value'),('#1','Adoption rate')],
        'deploy_model':[('Power BI Service','Cloud SaaS'),('Power BI Report Server','On-prem')],
        'source_compat':['Snowflake (DirectQuery)','SQL Server','Azure SQL','Excel','100+ sources'],
        'pricing':[('Pro','$10/user/mo'),('Premium Per User','$20/user/mo'),('Premium Capacity','$4,995+/mo')],
    },
    'ThoughtSpot': {'sf_fit':9,'ease':9,'enterprise':7,'cost':5,'tag':'AI-driven NLQ analytics',
        'strengths':['Natural language search','SpotIQ AI insights','Snowflake-native','Democratizes data'],
        'watchouts':['Expensive per-seat','Less custom viz','Needs well-modeled data'],
        'capabilities':['NLQ search','AI insights','Snowflake-native'],
        'key_facts':[('AI','Natural language'),('Minutes','Time to insight')],
        'deploy_model':[('Managed SaaS','Fully hosted')],
        'source_compat':['Snowflake','BigQuery','Redshift','Databricks'],
        'pricing':[('Essentials','Contact sales'),('Pro','Contact sales'),('Enterprise','Custom')],
    },
    'Looker': {'sf_fit':7,'ease':6,'enterprise':7,'cost':5,'tag':'LookML governed metrics',
        'strengths':['LookML for governed metrics','Google ecosystem','Strong modeling layer'],
        'watchouts':['Steep LookML learning curve','Google acquisition uncertainty','Expensive'],
        'capabilities':['LookML','Google Cloud','Governed metrics'],
        'key_facts':[('LookML','Modeling layer'),('Weeks','Time to value')],
        'deploy_model':[('Google Cloud','Managed SaaS')],
        'source_compat':['Snowflake','BigQuery','Redshift','Postgres'],
        'pricing':[('Standard','Contact sales'),('Enterprise','Custom ($3K+/mo)')],
    },
    'Metabase': {'sf_fit':6,'ease':8,'enterprise':4,'cost':9,'tag':'Free open-source BI',
        'strengths':['Free self-hosted','Simple, good for small teams','Quick dashboard creation'],
        'watchouts':['Limited enterprise governance','Self-hosted maintenance','Less polished'],
        'capabilities':['Open Source','Self-hosted','SQL-native'],
        'key_facts':[('Free','Self-hosted'),('Hours','Time to value')],
        'deploy_model':[('Self-hosted','Open source'),('Metabase Cloud','Managed')],
        'source_compat':['Snowflake','Postgres','MySQL','BigQuery'],
        'pricing':[('Open Source','$0'),('Pro','$85/10 users/mo'),('Enterprise','Custom')],
    },
    'Apache Superset': {'sf_fit':6,'ease':6,'enterprise':4,'cost':9,'tag':'Free open-source BI',
        'strengths':['Free','Developer-friendly','SQL-native'],
        'watchouts':['Complex setup','Limited self-service','Smaller community'],
        'capabilities':['Open Source','SQL-native','Developer-focused'],
        'key_facts':[('Free','Open source'),('Days','Time to value')],
        'deploy_model':[('Self-hosted','Docker / K8s'),('Preset.io','Managed SaaS')],
        'source_compat':['Snowflake','Most SQL databases'],
        'pricing':[('Open Source','$0'),('Preset.io','$20+/user/mo')],
    },
    'Streamlit': {'sf_fit':10,'ease':7,'enterprise':4,'cost':10,'tag':'Custom data apps, zero cost',
        'strengths':['Zero licensing, runs in Snowflake','Python-native','Great for custom apps/ML demos'],
        'watchouts':['Not a BI tool','Requires coding','No drag-and-drop dashboards'],
        'capabilities':['Snowflake-native','Python','Custom apps','Streamlit in Snowflake'],
        'key_facts':[('$0','Licensing cost'),('Hours','Time to build app')],
        'deploy_model':[('Streamlit in Snowflake','Native, zero infra'),('Streamlit Cloud','Managed')],
        'source_compat':['Snowflake (native)','Any Python-accessible source'],
        'pricing':[('Included','Warehouse credits only')],
    },
    'Apache Airflow / MWAA': {'sf_fit':8,'ease':5,'enterprise':7,'cost':7,'tag':'Complex multi-system orchestration',
        'strengths':['Most adopted orchestrator','Python-native, extensible','Snowflake Provider','Huge community'],
        'watchouts':['Steep learning curve','Heavy ops if self-hosted','Complex for non-engineers'],
        'capabilities':['Python DAGs','Multi-system','Extensible','Huge community'],
        'key_facts':[('10M+','Downloads/mo'),('Days','Time to value'),('#1','Orchestrator adoption')],
        'deploy_model':[('Self-hosted','Kubernetes / Docker'),('MWAA','AWS managed'),('Astronomer','Managed SaaS')],
        'source_compat':['Any system (via Operators)','Snowflake Provider','dbt integration','Cloud APIs'],
        'pricing':[('Self-hosted','$0 + infra'),('MWAA','~$0.49/hr (~$350+/mo)'),('Astronomer','$1,000+/mo')],
    },
    'Snowflake Tasks & Streams': {'sf_fit':10,'ease':8,'enterprise':6,'cost':10,'tag':'Zero-cost native scheduling',
        'strengths':['Zero additional cost','Native Snowflake','Streams for CDC','Serverless Tasks'],
        'watchouts':['Snowflake-only workflows','No visual DAG editor','No multi-system orchestration'],
        'capabilities':['Snowflake-native','Serverless','CDC Streams','Zero cost'],
        'key_facts':[('$0','Additional licensing'),('Minutes','Time to setup'),('Native','CDC via Streams')],
        'deploy_model':[('Serverless','Snowflake-managed'),('Native','No external infra')],
        'source_compat':['Snowflake (exclusive)'],
        'pricing':[('Included','Warehouse/serverless credits only')],
    },
    'Prefect': {'sf_fit':7,'ease':7,'enterprise':6,'cost':8,'tag':'Modern Python orchestrator',
        'strengths':['Modern DX, better than Airflow','Python-native','Hybrid execution'],
        'watchouts':['Smaller community','Less battle-tested','Fewer integrations'],
        'capabilities':['Python-native','Modern DX','Hybrid'],
        'key_facts':[('Modern','Developer experience'),('Hours','Time to value')],
        'deploy_model':[('Self-hosted','Open source'),('Prefect Cloud','Managed SaaS')],
        'source_compat':['Any Python-accessible system','Snowflake','dbt'],
        'pricing':[('Open Source','$0'),('Cloud','$500+/mo')],
    },
    'Dagster': {'sf_fit':7,'ease':7,'enterprise':6,'cost':8,'tag':'Asset-centric orchestration',
        'strengths':['Asset-centric paradigm','Built-in lineage','dbt integration'],
        'watchouts':['Smaller ecosystem','Learning curve','Newer'],
        'capabilities':['Asset-centric','Built-in lineage','dbt integration'],
        'key_facts':[('Asset','Centric paradigm'),('Hours','Time to value')],
        'deploy_model':[('Self-hosted','Open source'),('Dagster Cloud','Managed')],
        'source_compat':['Snowflake','dbt','Any Python source'],
        'pricing':[('Open Source','$0'),('Cloud','Custom')],
    },
    'dbt Cloud Scheduler': {'sf_fit':9,'ease':9,'enterprise':7,'cost':6,'tag':'Built-in dbt scheduling',
        'strengths':['Included with dbt Cloud','Zero setup','Slim CI'],
        'watchouts':['Only schedules dbt models','Needs dbt Cloud license','No multi-system orchestration'],
        'capabilities':['dbt-native','CI/CD','Slim CI'],
        'key_facts':[('Included','With dbt Cloud'),('Zero','Setup needed')],
        'deploy_model':[('dbt Cloud','Managed SaaS')],
        'source_compat':['dbt models only'],
        'pricing':[('Included','With dbt Cloud license')],
    },
    'Cron / Custom Scheduler': {'sf_fit':3,'ease':6,'enterprise':2,'cost':10,'tag':'Basic free scheduling',
        'strengths':['Free','Simple'],
        'watchouts':['No monitoring','No retry logic','No DAG dependencies'],
        'capabilities':['Basic','Free','Cron syntax'],
        'key_facts':[('Free','No licensing'),('Minutes','To configure')],
        'deploy_model':[('Any server','Linux / Mac / Cloud')],
        'source_compat':['Any callable command'],
        'pricing':[('Free','$0')],
    },
    'Control-M': {'sf_fit':5,'ease':4,'enterprise':8,'cost':3,'tag':'Legacy enterprise scheduler',
        'strengths':['Enterprise-grade','Mature'],
        'watchouts':['Expensive licensing','Legacy architecture','Hard to integrate with cloud'],
        'capabilities':['Enterprise','Cross-platform','Mainframe support'],
        'key_facts':[('Enterprise','Grade scheduling'),('Weeks','Time to value')],
        'deploy_model':[('On-prem','Server-based'),('SaaS','BMC Helix')],
        'source_compat':['Mainframe','SAP','Oracle','Any job type'],
        'pricing':[('Enterprise license','$50K-$500K+/yr')],
    },
    'Snowflake Horizon (native)': {'sf_fit':10,'ease':8,'enterprise':7,'cost':10,'tag':'Native governance, zero cost',
        'strengths':['Included with Snowflake','Masking, RLS, Tagging, Classification','Access History and lineage','Auto PII detection'],
        'watchouts':['Snowflake-only','Less rich UI than dedicated tools','No cross-platform catalog'],
        'capabilities':['Masking','RLS','Tagging','Classification','Lineage'],
        'key_facts':[('$0','Additional cost'),('Native','Built into Snowflake'),('Auto','PII detection')],
        'deploy_model':[('Native','Snowflake built-in')],
        'source_compat':['Snowflake (exclusive)'],
        'pricing':[('Included','With Snowflake edition')],
    },
    'Snowflake Horizon': {'sf_fit':10,'ease':8,'enterprise':7,'cost':10,'tag':'Native governance, zero cost',
        'strengths':['Included with Snowflake','Masking, RLS, Tagging, Classification','Access History and lineage'],
        'watchouts':['Snowflake-only','Less rich UI'],
        'capabilities':['Masking','RLS','Tagging','Lineage'],
        'key_facts':[('$0','Additional cost'),('Native','Built-in')],
        'deploy_model':[('Native','Snowflake built-in')],
        'source_compat':['Snowflake (exclusive)'],
        'pricing':[('Included','With Snowflake edition')],
    },
    'Monte Carlo': {'sf_fit':9,'ease':8,'enterprise':8,'cost':5,'tag':'ML-powered data observability',
        'strengths':['Automated ML anomaly detection','End-to-end lineage','Incident management','Snowflake-native'],
        'watchouts':['Expensive per-table','False positives initially','Tuning period'],
        'capabilities':['ML anomaly detection','Lineage','Incident mgmt','Snowflake-native'],
        'key_facts':[('ML','Anomaly detection'),('Minutes','Alert latency'),('E2E','Lineage coverage')],
        'deploy_model':[('Managed SaaS','Fully hosted'),('Snowflake-native','In-account agent')],
        'source_compat':['Snowflake','BigQuery','Redshift','Databricks','dbt'],
        'pricing':[('Starter','Contact sales'),('Enterprise','Custom (~$5K-$30K+/mo)')],
    },
    'Atlan': {'sf_fit':8,'ease':9,'enterprise':7,'cost':5,'tag':'Modern collaborative catalog',
        'strengths':['Modern UI, Slack-like','Automated lineage','AI-powered search','Strong Snowflake integration'],
        'watchouts':['Expensive for small teams','Less mature for compliance'],
        'capabilities':['Data catalog','Auto lineage','AI search','Collaboration'],
        'key_facts':[('AI','Powered search'),('Auto','Lineage discovery'),('Slack-like','Collaboration UX')],
        'deploy_model':[('Managed SaaS','Fully hosted')],
        'source_compat':['Snowflake','BigQuery','Redshift','dbt','Airflow','Tableau'],
        'pricing':[('Team','Contact sales'),('Enterprise','Custom (~$3K-$15K+/mo)')],
    },
    'Great Expectations': {'sf_fit':6,'ease':5,'enterprise':4,'cost':10,'tag':'Open-source Python DQ',
        'strengths':['Free','Python-native','Extensive expectations library'],
        'watchouts':['Steep learning curve','Requires Python','Operationally heavy'],
        'capabilities':['Open Source','Python-native','Data quality'],
        'key_facts':[('Free','Open source'),('300+','Expectations'),('Python','Native')],
        'deploy_model':[('Self-hosted','Python library'),('GX Cloud','Managed')],
        'source_compat':['Snowflake','Any Python-accessible source'],
        'pricing':[('Open Source','$0'),('GX Cloud','Custom')],
    },
    'Soda': {'sf_fit':7,'ease':8,'enterprise':5,'cost':8,'tag':'YAML-based DQ checks',
        'strengths':['Simple YAML checks','dbt + Airflow integration','Open-source core'],
        'watchouts':['Less automated than Monte Carlo','No ML anomaly detection in free tier'],
        'capabilities':['YAML checks','dbt integration','Open Source core'],
        'key_facts':[('YAML','Simple syntax'),('dbt','Native integration')],
        'deploy_model':[('CLI','Open source'),('Soda Cloud','Managed')],
        'source_compat':['Snowflake','BigQuery','Postgres','Redshift'],
        'pricing':[('Soda Core','$0'),('Soda Cloud','$200+/mo')],
    },
    'Snowflake Horizon Catalog (native)': {'sf_fit':10,'ease':8,'enterprise':6,'cost':10,'tag':'Native Snowflake catalog',
        'strengths':['Included with Snowflake','Native tagging and classification'],
        'watchouts':['Snowflake-only','Less rich than Atlan/Collibra'],
        'capabilities':['Tagging','Classification','Native'],
        'key_facts':[('$0','Cost'),('Native','Built-in')],
        'deploy_model':[('Native','Snowflake built-in')],
        'source_compat':['Snowflake (exclusive)'],
        'pricing':[('Included','With Snowflake')],
    },
    'Collibra': {'sf_fit':7,'ease':5,'enterprise':9,'cost':3,'tag':'Enterprise governance leader',
        'strengths':['Industry-leading governance','Business glossary & stewardship','Regulatory compliance workflows','Broad ecosystem integration'],
        'watchouts':['Very expensive','Complex implementation','Steep learning curve'],
        'capabilities':['Governance','Glossary','Stewardship','Compliance'],
        'key_facts':[('#1','Governance platform'),('Months','Time to value')],
        'deploy_model':[('Cloud','Managed SaaS'),('Hybrid','On-prem option')],
        'source_compat':['Snowflake','All major databases','BI tools','ETL tools'],
        'pricing':[('Enterprise','Custom ($50K-$500K+/yr)')],
    },
    'DataHub (LinkedIn OSS)': {'sf_fit':6,'ease':5,'enterprise':5,'cost':9,'tag':'Open-source data catalog',
        'strengths':['Free, open-source','Strong lineage','Active community'],
        'watchouts':['Complex setup','Requires engineering resources'],
        'capabilities':['Open Source','Lineage','Metadata'],
        'key_facts':[('Free','Open source'),('Strong','Lineage')],
        'deploy_model':[('Self-hosted','Kubernetes'),('Acryl Data','Managed')],
        'source_compat':['Snowflake','Most databases','dbt','Airflow'],
        'pricing':[('Open Source','$0'),('Acryl Data','Custom')],
    },
    'Snowflake Native Ingestion': {'sf_fit':10,'ease':8,'enterprise':7,'cost':10,'tag':'Built-in, zero extra cost',
        'strengths':['COPY INTO for bulk loads','Snowpipe for auto-ingest','Snowpipe Streaming for real-time','Zero external dependencies'],
        'watchouts':['Requires S3/Azure Blob/GCS staging','No built-in SaaS connectors','Manual setup for CDC'],
        'capabilities':['COPY INTO','Snowpipe','Snowpipe Streaming','Zero cost'],
        'key_facts':[('$0','Additional licensing'),('Seconds','Snowpipe Streaming latency'),('Native','No external tools')],
        'deploy_model':[('Native','Snowflake built-in'),('Serverless','Snowpipe')],
        'source_compat':['S3 / Azure Blob / GCS','Kafka (connector)','Files (CSV, JSON, Parquet, etc.)'],
        'pricing':[('Included','Warehouse/serverless credits only')],
    },
    'Rivery': {'sf_fit':7,'ease':8,'enterprise':5,'cost':6,'tag':'Mid-market SaaS ELT',
        'strengths':['Easy setup','SaaS connectors','Built-in orchestration'],
        'watchouts':['Fewer advanced features','Smaller community','Per-pipeline pricing'],
        'capabilities':['ELT','Managed SaaS','Orchestration'],
        'key_facts':[('150+','Connectors'),('Hours','Time to value')],
        'deploy_model':[('Managed SaaS','Fully hosted')],
        'source_compat':['SaaS apps','Cloud DBs','APIs'],
        'pricing':[('Starter','~$500/mo'),('Professional','~$2,000/mo'),('Enterprise','Custom')],
    },
}

def _build_tool_name_index(db):
    index = {}
    for name in db:
        index[name.lower()] = name
        for word in name.replace('(', '').replace(')', '').replace('/', ' ').split():
            if len(word) > 3:
                if word.lower() not in index:
                    index[word.lower()] = name
    return index

_tool_index = _build_tool_name_index(TOOL_DB)

def _resolve_tool(card_name, db, index):
    if card_name in db:
        return card_name
    if card_name.lower() in index:
        return index[card_name.lower()]
    parts = card_name.replace('(', '').replace(')', '').replace('/', ' ').split()
    for part in parts:
        if part.lower() in index:
            return index[part.lower()]
    return None

tc_data = plan.get('tool_comparisons', [])
unmatched = []
for section in tc_data:
    for card in section.get('cards', []):
        resolved = _resolve_tool(card['name'], TOOL_DB, _tool_index)
        if resolved and resolved in TOOL_DB:
            db = TOOL_DB[resolved]
            for k in ['sf_fit','ease','enterprise','cost','tag','strengths','watchouts','capabilities','key_facts','deploy_model','source_compat','pricing']:
                if k not in card:
                    card[k] = db[k]
        else:
            unmatched.append(card['name'])
            for k in ['tag','strengths','watchouts','capabilities','key_facts','deploy_model','source_compat','pricing']:
                if k not in card: card[k] = [] if k != 'tag' else ''

print(f"Tool comparisons: {len(tc_data)} categories enriched from TOOL_DB")
if unmatched:
    print(f"WARNING: {len(unmatched)} card(s) not found in TOOL_DB: {unmatched}")

In [ ]:
CSS = """
@page { size: A4; margin: 18mm 16mm; }
@media print {
  body { -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important; }
  .page-break { page-break-before: always; }
  .no-break { page-break-inside: avoid; }
}
:root {
  --g: #B3D943; --gl: #E8F5C8; --gd: #4A5A2B;
  --bk: #1A1A1A; --nv: #2C3E6B; --cy: #29B5E8; --lb: #7ECCE5;
  --dg: #333; --mg: #666; --lg: #F5F5F5; --bg: #E0E0E0; --w: #FFF;
  --rc: #D32F2F; --rh: #F57C00; --rm: #666; --rs: #388E3C;
  --teal: #00B4D8; --orange: #FF6B35;
}
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: 'Proxima Nova','Helvetica Neue',Helvetica,Arial,sans-serif; color: var(--dg); font-size: 11pt; line-height: 1.6; max-width: 210mm; margin: 0 auto; }
.cover { display:flex; flex-direction:column; align-items:center; justify-content:center; min-height:92vh; text-align:center; }
.cover .brand { font-size:13pt; color:var(--g); font-weight:600; letter-spacing:2px; }
.cover .divider { width:60px; height:2px; background:var(--g); margin:20px auto; }
.cover h1 { font-size:32pt; color:var(--bk); font-weight:700; line-height:1.15; margin:4px 0; }
.cover .customer { font-size:17pt; color:var(--mg); font-weight:400; margin:4px 0 16px; }
.cover .meta-table td { padding:4px 12px; font-size:11pt; }
.cover .meta-table td:first-child { color:var(--mg); text-align:right; font-weight:400; }
.cover .meta-table td:last-child { color:var(--bk); font-weight:600; text-align:left; }
.cover .confidential { font-size:9pt; color:#999; font-style:italic; margin-top:40px; }
h2 { font-size:16pt; color:var(--bk); font-weight:700; margin:28px 0 12px; padding-bottom:4px; border-bottom:2px solid var(--g); display:inline-block; }
h3 { font-size:12pt; color:var(--bk); font-weight:600; margin:16px 0 8px; }
.sec { margin-bottom:20px; }
.metrics { display:grid; grid-template-columns:repeat(4,1fr); gap:10px; margin:12px 0 16px; }
.metric-card { background:var(--lg); border-radius:6px; padding:14px 12px; text-align:center; border-top:3px solid var(--g); }
.metric-card .val { font-size:22pt; font-weight:700; color:var(--bk); line-height:1.2; }
.metric-card .lbl { font-size:8.5pt; color:var(--mg); letter-spacing:0.5px; margin-top:4px; }
.tag { display:inline-block; padding:3px 10px; border-radius:4px; font-size:9pt; font-weight:600; margin:2px 3px; }
.tag-green { background:var(--gl); color:var(--gd); }
.tag-blue { background:#E3F2FD; color:var(--nv); }
.tag-gray { background:var(--lg); color:var(--dg); }
.tag-teal { background:#E0F7FA; color:#00838F; }
.tag-orange { background:#FFF3E0; color:#BF360C; }
table { width:100%; border-collapse:collapse; margin:8px 0 14px; font-size:10pt; }
th { background:var(--bk); color:var(--w); padding:8px 10px; text-align:left; font-weight:600; font-size:9pt; letter-spacing:0.3px; }
td { padding:7px 10px; border-bottom:1px solid var(--bg); }
tr:nth-child(even) td { background:var(--lg); }
.timeline { position:relative; margin:14px 0 14px 24px; padding-left:24px; border-left:2px solid var(--g); }
.tl-item { position:relative; margin-bottom:18px; }
.tl-item::before { content:''; position:absolute; left:-30px; top:4px; width:12px; height:12px; border-radius:50%; background:var(--g); border:2px solid var(--w); }
.tl-item .tl-head { font-size:11pt; font-weight:700; color:var(--bk); }
.tl-item .tl-sub { font-size:9pt; color:var(--mg); margin-bottom:4px; }
.tl-item ul { margin:4px 0 0 16px; font-size:10pt; }
.tl-item li { margin-bottom:3px; }
.risk-grid { display:grid; grid-template-columns:1fr 1fr; gap:8px; margin:10px 0; }
.risk-card { padding:10px 12px; border-radius:5px; border-left:4px solid; font-size:10pt; }
.risk-card.critical { border-color:var(--rc); background:#FFF5F5; }
.risk-card.high { border-color:var(--rh); background:#FFF8F0; }
.risk-card.medium { border-color:var(--mg); background:var(--lg); }
.risk-card .risk-title { font-weight:700; font-size:10pt; }
.risk-card .risk-mit { color:var(--mg); font-size:9pt; margin-top:3px; }
.kpi-row { display:flex; align-items:center; padding:8px 0; border-bottom:1px solid var(--bg); font-size:10pt; }
.kpi-row .kpi-name { width:35%; font-weight:600; color:var(--bk); }
.kpi-row .kpi-target { width:35%; color:var(--dg); }
.kpi-row .kpi-method { width:30%; color:var(--mg); font-size:9pt; }
.schema-flow { display:flex; gap:8px; margin:10px 0; flex-wrap:wrap; }
.schema-box { flex:1; min-width:90px; background:var(--lg); border-top:3px solid var(--g); border-radius:5px; padding:10px; text-align:center; }
.schema-box .s-name { font-size:10pt; font-weight:700; color:var(--bk); }
.schema-box .s-desc { font-size:8pt; color:var(--mg); margin-top:3px; }
.schema-arrow { display:flex; align-items:center; color:var(--g); font-size:16pt; font-weight:700; }
.comp-grid { display:grid; grid-template-columns:repeat(3,1fr); gap:8px; margin:10px 0; }
.comp-card { background:var(--lg); border-radius:5px; padding:10px 12px; }
.comp-card .comp-req { font-size:10pt; font-weight:700; color:var(--bk); }
.comp-card .comp-impl { font-size:8.5pt; color:var(--mg); margin-top:3px; }
.steps ol { counter-reset:step; list-style:none; margin:10px 0; padding:0; }
.steps li { counter-increment:step; display:flex; align-items:flex-start; margin-bottom:8px; font-size:10.5pt; }
.steps li::before { content:counter(step); background:var(--g); color:var(--w); font-weight:700; font-size:9pt; width:24px; height:24px; border-radius:50%; display:flex; align-items:center; justify-content:center; margin-right:10px; flex-shrink:0; }

/* === NEW TOOL PROFILE CARDS === */
.tc-cards { display:grid; grid-template-columns:repeat(auto-fit, minmax(220px, 1fr)); gap:12px; margin:12px 0; }
.tc-card { background:var(--w); border:1px solid #E0E0E0; border-radius:8px; padding:18px; font-size:9pt; }
.tc-card.recommended { border:2px solid var(--teal); box-shadow:0 2px 8px rgba(0,180,216,0.12); }
.tc-card .tc-header { margin-bottom:10px; }
.tc-card .tc-name { font-size:13pt; font-weight:700; color:var(--bk); line-height:1.3; }
.tc-card .tc-tagline { font-size:8pt; color:var(--mg); margin:2px 0 6px; }
.tc-card .tc-badges { display:flex; flex-wrap:wrap; gap:4px; margin:6px 0 12px; }
.tc-card .tc-badge { display:inline-block; padding:2px 8px; border-radius:3px; font-size:7.5pt; font-weight:600; background:#F0F0F0; color:var(--dg); }
.tc-card .tc-badge.native { background:#E0F7FA; color:#00838F; }
.tc-card .tc-badge.rec-badge { background:var(--teal); color:var(--w); }
.tc-card .tc-section-label { font-size:7.5pt; font-weight:700; color:var(--mg); letter-spacing:0.6px; text-transform:uppercase; margin:12px 0 6px; border-bottom:1px solid #EEE; padding-bottom:3px; }
.tc-card .tc-scores { margin:6px 0; }
.tc-card .tc-score-row { display:flex; align-items:center; margin:4px 0; }
.tc-card .tc-score-label { width:52%; font-size:8pt; color:var(--mg); }
.tc-card .tc-score-bar-bg { width:34%; height:7px; background:#EDEDED; border-radius:4px; overflow:hidden; margin:0 6px; }
.tc-card .tc-score-bar { height:100%; border-radius:4px; transition:width 0.3s; }
.tc-card .tc-score-val { width:14%; text-align:right; font-size:9pt; font-weight:700; color:var(--bk); }
.tc-card .tc-facts { display:grid; grid-template-columns:1fr 1fr; gap:6px; margin:6px 0; }
.tc-card .tc-fact { background:var(--lg); border-radius:5px; padding:8px; text-align:center; }
.tc-card .tc-fact .tc-fact-val { font-size:13pt; font-weight:700; color:var(--bk); line-height:1.2; }
.tc-card .tc-fact .tc-fact-lbl { font-size:7pt; color:var(--mg); margin-top:1px; }
.tc-card .tc-deploy { display:flex; flex-wrap:wrap; gap:4px; margin:6px 0; }
.tc-card .tc-deploy-chip { padding:3px 8px; border-radius:4px; font-size:7.5pt; font-weight:600; }
.tc-card .tc-deploy-chip.primary { background:var(--gl); color:var(--gd); }
.tc-card .tc-deploy-chip.secondary { background:#E3F2FD; color:var(--nv); }
.tc-card .tc-compat-list { list-style:none; padding:0; margin:4px 0; }
.tc-card .tc-compat-list li { font-size:8pt; color:var(--dg); padding:1.5px 0; }
.tc-card .tc-compat-list li::before { content:'\25B8'; color:var(--teal); margin-right:5px; font-size:7pt; }
.tc-card .tc-pricing-tbl { width:100%; border-collapse:collapse; margin:4px 0; }
.tc-card .tc-pricing-tbl td { padding:3px 0; font-size:8pt; border-bottom:1px solid #F0F0F0; }
.tc-card .tc-pricing-tbl td:first-child { color:var(--dg); font-weight:500; }
.tc-card .tc-pricing-tbl td:last-child { text-align:right; color:var(--bk); font-weight:600; }
.tc-card .tc-pricing-note { font-size:7pt; color:#999; font-style:italic; margin-top:4px; }
.tc-card .sw-section { margin-top:8px; }
.tc-card .sw-item { display:flex; align-items:flex-start; font-size:8pt; margin:2.5px 0; line-height:1.35; }
.tc-card .sw-dot { width:5px; height:5px; border-radius:50%; margin:4px 5px 0 0; flex-shrink:0; }
.tc-card .sw-dot.green { background:var(--rs); }
.tc-card .sw-dot.amber { background:var(--rh); }

.verdict { background:var(--gl); border-left:3px solid var(--g); padding:10px 14px; margin:10px 0; border-radius:0 5px 5px 0; }
.verdict .v-label { font-size:8pt; font-weight:700; color:var(--gd); letter-spacing:0.5px; margin-bottom:2px; }
.verdict .v-text { font-size:9.5pt; color:var(--bk); }

.tc-legend { display:flex; gap:16px; margin:6px 0 14px; font-size:8pt; color:var(--mg); }
.tc-legend-item { display:flex; align-items:center; gap:4px; }
.tc-legend-dot { width:8px; height:8px; border-radius:50%; }
.tc-legend-dot.g { background:var(--rs); }
.tc-legend-dot.a { background:var(--rh); }

p { margin:4px 0 8px; font-size:10.5pt; }
ul { margin:4px 0 10px 18px; } li { margin-bottom:3px; font-size:10pt; }
.footer-note { font-size:8.5pt; color:#999; font-style:italic; margin-top:28px; padding-top:10px; border-top:1px solid var(--bg); }
"""
print('CSS ready (kipi.ai brand v8 — new tool profile cards).')

In [ ]:
def tbl(headers, rows):
    h = ''.join(f'<th>{c}</th>' for c in headers)
    b = ''.join('<tr>' + ''.join(f'<td>{c}</td>' for c in r) + '</tr>' for r in rows)
    return f'<table><thead><tr>{h}</tr></thead><tbody>{b}</tbody></table>'

def metric_card(val, label):
    return f'<div class="metric-card no-break"><div class="val">{val}</div><div class="lbl">{label}</div></div>'

def tag(text, cls='green'):
    return f'<span class="tag tag-{cls}">{text}</span>'

def risk_card(risk, sev, mit):
    cls = {'CRITICAL':'critical','High':'high'}.get(sev,'medium')
    return f'<div class="risk-card {cls} no-break"><div class="risk-title">{sev}: {risk}</div><div class="risk-mit">{mit}</div></div>'

def score_bar_color(score):
    if score >= 8: return '#2ECC71'
    if score >= 6: return '#F39C12'
    return '#E74C3C'

def tool_comp_card(t, is_rec=False):
    rec_cls = ' recommended' if is_rec else ''
    h = f'<div class="tc-card{rec_cls} no-break">'
    h += '<div class="tc-header">'
    h += f'<div class="tc-name">{t["name"]}</div>'
    h += f'<div class="tc-tagline">{t.get("tag","")}</div>'
    h += '<div class="tc-badges">'
    if is_rec:
        h += '<span class="tc-badge rec-badge">&#10004; Recommended</span>'
    for cap in t.get('capabilities', [])[:4]:
        native = ' native' if any(kw in cap.lower() for kw in ['native','snowflake','open source']) else ''
        h += f'<span class="tc-badge{native}">{cap}</span>'
    h += '</div></div>'
    h += '<div class="tc-section-label">Capability Scores /10</div>'
    h += '<div class="tc-scores">'
    score_labels = [('sf_fit','Snowflake fit'),('ease','Ease of adoption'),('enterprise','Enterprise depth'),('cost','Cost efficiency')]
    for k, lbl in score_labels:
        if k in t:
            score = t[k]
            pct = score * 10
            color = score_bar_color(score)
            h += f'<div class="tc-score-row"><span class="tc-score-label">{lbl}</span>'
            h += f'<div class="tc-score-bar-bg"><div class="tc-score-bar" style="width:{pct}%;background:{color}"></div></div>'
            h += f'<span class="tc-score-val">{score:.1f}</span></div>'
    h += '</div>'
    key_facts = t.get('key_facts', [])
    if key_facts:
        h += '<div class="tc-section-label">Key Facts</div><div class="tc-facts">'
        for val, lbl in key_facts[:4]:
            h += f'<div class="tc-fact"><div class="tc-fact-val">{val}</div><div class="tc-fact-lbl">{lbl}</div></div>'
        h += '</div>'
    deploy = t.get('deploy_model', [])
    if deploy:
        h += '<div class="tc-deploy">'
        for i, (label, desc) in enumerate(deploy[:3]):
            cls = 'primary' if i == 0 else 'secondary'
            h += f'<span class="tc-deploy-chip {cls}" title="{desc}">{label}</span>'
        h += '</div>'
    compat = t.get('source_compat', [])
    if compat:
        h += '<div class="tc-section-label">Source Compatibility</div><ul class="tc-compat-list">'
        for src in compat[:6]:
            h += f'<li>{src}</li>'
        h += '</ul>'
    pricing = t.get('pricing', [])
    if pricing:
        h += '<div class="tc-section-label">Pricing (USD/Month)</div><table class="tc-pricing-tbl">'
        for tier, cost in pricing[:4]:
            h += f'<tr><td>{tier}</td><td>{cost}</td></tr>'
        h += '</table>'
    h += '<div class="sw-section">'
    for s in t.get('strengths', [])[:3]:
        h += f'<div class="sw-item"><div class="sw-dot green"></div><span>{s}</span></div>'
    for w in t.get('watchouts', [])[:2]:
        h += f'<div class="sw-item"><div class="sw-dot amber"></div><span>{w}</span></div>'
    h += '</div></div>'
    return h

es = plan['executive_summary']
cs = plan['current_state']
ta = plan['target_architecture']
rt = plan['recommended_tools']
sc = plan['security_compliance']

sa_line = meta['sa_name']
if meta.get('sa_email'): sa_line += f" | {meta['sa_email']}"
sources = [s['name'] for s in cs['source_systems']]
compliance = [c for c in plan.get('compliance_list', [])]
if not compliance:
    for m in es.get('key_metrics', []):
        if m['metric'] == 'Compliance' and m['value'] != 'N/A':
            compliance = [c.strip() for c in m['value'].split(',')]
drivers = [d.strip() for d in es.get('drivers', '').replace('Drivers: ', '').rstrip('.').split(',') if d.strip()]
src_tags = ' '.join(tag(s, 'blue') for s in sources)
comp_tags = ' '.join(tag(c, 'gray') for c in compliance) if compliance else tag('N/A', 'gray')
driver_tags = ' '.join(tag(d, 'green') for d in drivers)
km = {m['metric']: m['value'] for m in es.get('key_metrics', [])}
sp_display = km.get('Stored Procedures', km.get('Stored Procedures / Packages', '0'))

html = f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Migration Plan - {meta['customer_name']}</title><style>{CSS}</style></head><body>
<div class="cover"><div class="brand">kipi.ai &bull; CoCoDuo</div><div class="divider"></div>
<h1>Migration Plan</h1><div class="customer">{meta['customer_name']}</div><div class="divider"></div>
<table class="meta-table"><tbody><tr><td>Solutions Architect</td><td>{sa_line}</td></tr>
<tr><td>Date</td><td>{meta['generated_date']}</td></tr>
<tr><td>Assessment</td><td>{meta['assessment_id']}</td></tr></tbody></table>
<div class="confidential">Confidential &mdash; Prepared by kipi.ai for {meta['customer_name']}</div></div>
<div class="page-break"></div>

<div class="sec"><h2>Executive Summary</h2>
<p>{es['overview']}</p>
<p>{es['drivers']}</p>
<p>{es['approach']}</p>"""
if es.get('reference'): html += f'<p><em>{es["reference"]}</em></p>'
html += '</div>'

html += f"""<div class="sec"><h2>Snapshot</h2>
<div class="metrics">{metric_card(km.get('Total Data Volume','N/A').split(' ')[0],'Total Data (TB)')}{metric_card(km.get('Total Tables','0'),'Tables')}{metric_card(sp_display,'Stored Procedures')}{metric_card(km.get('Active Users','0'),'Users')}</div>
<div class="metrics">{metric_card(km.get('Daily Queries','0'),'Daily Queries')}{metric_card(km.get('CDC Tables','0'),'CDC Tables')}{metric_card(km.get('Timeline','N/A'),'Timeline')}{metric_card(km.get('Budget Sensitivity','N/A'),'Budget')}</div>
<p><strong>Sources:</strong> {src_tags}</p><p><strong>Compliance:</strong> {comp_tags}</p><p><strong>Drivers:</strong> {driver_tags}</p></div>
<div class="page-break"></div>"""

html += f"""<div class="sec"><h2>Current State</h2><h3>Source Systems</h3>
{tbl(['System','Version','Size','Tables','Complexity'], [[s['name'],s.get('version',''),s.get('size',''),s.get('tables',''),s.get('complexity','')] for s in cs['source_systems']])}
<h3>ETL Tools</h3>{tbl(['Tool','Pipelines','Satisfaction','Action','Notes'], [[t['tool'],t.get('pipelines','-'),t.get('satisfaction','-'),t.get('action','-'),t.get('notes','')] for t in cs['tool_landscape']])}"""
if cs.get('pain_points'):
    html += '<h3>Key Pain Points</h3><ul>' + ''.join(f'<li>{p}</li>' for p in cs['pain_points']) + '</ul>'
html += '</div><div class="page-break"></div>'

html += f'<div class="sec"><h2>Target Architecture</h2><p>{tag(ta["account_edition"]["edition"],"green")} on {tag(ta["account_edition"].get("cloud_region",""),"blue")} &mdash; {ta["account_edition"].get("account_structure","")}</p>'
html += f'<p><strong>Justification:</strong> {ta["account_edition"].get("justification","")}</p>'
html += f'<h3>{ta["data_architecture"]["pattern"]}</h3><p>{ta["data_architecture"].get("description","")}</p><div class="schema-flow">'
for i, s in enumerate(ta['data_architecture'].get('schemas', [])):
    if i > 0: html += '<div class="schema-arrow">&rarr;</div>'
    html += f'<div class="schema-box"><div class="s-name">{s["name"]}</div><div class="s-desc">{s.get("purpose","")}</div></div>'
html += '</div>'
if ta.get('multi_tenancy') and ta['multi_tenancy'].get('pattern','N/A') != 'N/A':
    html += f'<p><strong>Multi-Tenancy:</strong> {tag(ta["multi_tenancy"]["pattern"],"blue")} &mdash; {ta["multi_tenancy"].get("description","")}</p>'
if ta.get('streaming_architecture'):
    html += f'<h3>Streaming Architecture</h3><p>{ta["streaming_architecture"]}</p>'
if ta.get('data_sharing'):
    html += f'<h3>Data Sharing & Marketplace</h3><p>{ta["data_sharing"]}</p>'
if ta.get('archival'):
    html += f'<h3>Archival Strategy</h3><p>{ta["archival"]}</p>'
html += '<h3>Warehouse Strategy</h3>' + tbl(['Warehouse','Size','Suspend','Purpose','Est $/mo'], [[w['name'],w['size'],w.get('auto_suspend',''),w['purpose'],w.get('est_monthly','-')] for w in ta['warehouses']]) + '</div><div class="page-break"></div>'

for section in tc_data:
    html += f'<div class="sec tc-section"><h2>{section["category"]}</h2>'
    html += '<div class="tc-legend"><span class="tc-legend-item"><span class="tc-legend-dot g"></span> Strength</span><span class="tc-legend-item"><span class="tc-legend-dot a"></span> Watch-out</span></div>'
    html += '<div class="tc-cards">'
    for card in section['cards']:
        html += tool_comp_card(card, is_rec=(card['name'] == section.get('recommended','')))
    html += f'</div><div class="verdict"><div class="v-label">Our Recommendation for {meta["customer_name"]}</div><div class="v-text"><strong>{section.get("recommended","")}</strong> &mdash; {section.get("verdict","")}</div></div></div><div class="page-break"></div>'

html += '<div class="sec"><h2>Recommended Tool Stack</h2>' + tbl(['Category','Tool','Justification','Est Annual Cost'], [[t['category'],t['tool'],t.get('justification',''),t.get('est_annual_cost','-')] for t in rt['tools']]) + '</div><div class="page-break"></div>'

if rt.get('cost_comparison'):
    cc = rt['cost_comparison']
    html += '<div class="sec"><h2>TCO Comparison</h2>'
    html += f'<div class="metrics">{metric_card(cc["total_current"],"Current Annual Cost")}{metric_card(cc["total_future"],"Projected Annual Cost")}{metric_card(cc["savings"],"Annual Savings")}{metric_card(cc["savings_pct"],"Cost Reduction")}</div>'
    html += '<h3>Current State Costs</h3>' + tbl(['Item','Annual Cost'], [[i['item'],i['cost']] for i in cc['current_items']])
    html += '<h3>Future State Costs</h3>' + tbl(['Item','Projected Annual Cost'], [[i['item'],i['cost']] for i in cc['future_items']])
    html += '</div><div class="page-break"></div>'

html += '<div class="sec"><h2>Migration Timeline</h2><div class="timeline">'
for ph in plan['migration_phases']:
    html += f'<div class="tl-item no-break"><div class="tl-head">Phase {ph["phase_num"]}: {ph["title"]}</div><div class="tl-sub">{ph["weeks"]}</div><ul>'
    for task in ph['tasks']: html += f'<li>{task}</li>'
    html += '</ul></div>'
html += '</div></div><div class="page-break"></div>'

if plan.get('performance_slas'):
    html += '<div class="sec"><h2>Performance SLAs</h2>' + tbl(['KPI','Current','Target SLA','Approach'], [[s['kpi'],s.get('current',''),s['target'],s.get('approach','')] for s in plan['performance_slas']]) + '</div><div class="page-break"></div>'

html += '<div class="sec"><h2>Security &amp; Compliance</h2><div class="comp-grid">'
for s in sc: html += f'<div class="comp-card no-break"><div class="comp-req">{s["requirement"]}</div><div class="comp-impl">{s["implementation"]}</div></div>'
html += '</div></div><div class="page-break"></div>'

html += '<div class="sec"><h2>Risks &amp; Mitigations</h2><div class="risk-grid">'
for r in plan['risks']: html += risk_card(r['risk'], r['severity'], r['mitigation'])
html += '</div></div><div class="page-break"></div>'

html += '<div class="sec"><h2>Success Criteria</h2>'
for s in plan['success_criteria']: html += f'<div class="kpi-row"><div class="kpi-name">{s["kpi"]}</div><div class="kpi-target">{s["target"]}</div><div class="kpi-method">{s.get("measurement","")}</div></div>'
html += '</div><div class="page-break"></div>'

html += '<div class="sec steps"><h2>Next Steps</h2><ol>'
for ns in plan['next_steps']: html += f'<li><span>{ns}</span></li>'
html += '</ol></div>'

ref_archs = plan.get('reference_architectures', [])
if ref_archs:
    html += '<div class="page-break"></div><div class="sec"><h2>Reference Architectures</h2>'
    for ref in ref_archs:
        html += f'<div class="no-break" style="margin-bottom:16px;"><h3>{ref["name"]} ({ref.get("industry","")})</h3><p>{ref.get("summary","")}</p>'
        html += f'<p><strong>Lessons Learned:</strong> {ref.get("lessons","")}</p>'
        if ref.get('metrics'):
            html += '<table><thead><tr><th>Metric</th><th>Result</th></tr></thead><tbody>'
            for mk, mv in ref['metrics'].items():
                html += f'<tr><td>{mk.replace("_"," ").title()}</td><td>{mv}</td></tr>'
            html += '</tbody></table>'
        html += '</div>'
    html += '</div>'

html += f'<div class="footer-note">Generated by {meta["generated_by"]} | {meta["assessment_id"]} | {meta["generated_date"]}<br>This document is confidential and intended solely for {meta["customer_name"]}.</div></body></html>'

safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', meta['customer_name'])
tmp_path = f'/tmp/{safe_name}_Migration_Plan.html'
with open(tmp_path, 'w') as f: f.write(html)
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("CREATE STAGE IF NOT EXISTS MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') DIRECTORY = (ENABLE = TRUE)").collect()
session.file.put(tmp_path, '@MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS', auto_compress=False, overwrite=True)
print(f'Plan generated: {os.path.getsize(tmp_path)/1024:.1f} KB')
print(f'Sections: Executive Summary, Snapshot, Current State, Target Architecture, Tool Comparisons ({len(tc_data)}), Recommended Stack, TCO, Timeline ({len(plan["migration_phases"])} phases), Performance SLAs ({len(plan.get("performance_slas",[]))}), Security & Compliance, Risks ({len(plan["risks"])}), Success Criteria, Next Steps, Reference Architectures ({len(ref_archs)})')
print(f'\nTo download:')
print(f"  SELECT GET_PRESIGNED_URL(@MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS, '{safe_name}_Migration_Plan.html', 3600);")
from IPython.display import HTML as IPyHTML, display
display(IPyHTML(f'<div style="padding:12px;background:#E8F5C8;border-left:3px solid #B3D943;border-radius:4px;font-size:13px;"><strong>Migration Plan Generated</strong> for {meta["customer_name"]}<br>Run the presigned URL query above to view/download. Ctrl+P to save as PDF.</div>'))